In [2]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA, AirQuality
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from pytorch_lightning.loggers import TensorBoardLogger



def find_cliques_size_k(G, k):
    all_cliques = set()
    for clique in nx.find_cliques(G):
        if len(clique) == k:
            all_cliques.add(tuple(sorted(clique)))
        elif len(clique) > k:
            for mini_clique in itertools.combinations(clique, k):
                all_cliques.add(tuple(sorted(mini_clique)))
    return list(all_cliques)




def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision('medium')  # 'medium' favors performance over precision

        # Enable TF32 format which is optimized for Tensor Cores on Ampere+ GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(42)

42

In [3]:
dataset = MetrLA(root='./data/metrla')

connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        # normalize_axis=1,
                                        force_symmetric=False,
                                        layout="edge_index")

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                      mask=dataset.mask,
                                      covariates=covariates,
                                      horizon=12,
                                      window=12,
                                      stride=1)
print(torch_dataset)

SpatioTemporalDataset(n_samples=34249, n_nodes=207, n_channels=1)


/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:98: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(df.index[0], df.index[-1], freq='5T')
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:109: FutureWarning: The 'method' keyword in DataFrame.replace is deprecated and will be removed in a future version.
  df = df.replace(to_replace=0., method='ffill')


In [4]:
# dataset = AirQuality(root='./data/aq36', impute_nans=True, small=True)

# splitting = {"val_len": 0.1,
#             "test_len": 0.2}


# connectivity_sparse= {"method": "distance",
#                     "threshold": 0.8,
#                     "include_self": False,
#                     "layout": "edge_index",
#                      "normalize_axis":1}

# adj = dataset.get_connectivity(**connectivity_sparse)

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = HO_Pre(target=dataset.dataframe(),
#                                       connectivity=adj,
#                                       mask=dataset.mask,
#                                       covariates=covariates,
#                                       horizon=12,
#                                       window=12,
#                                       stride=1)

# torch_dataset

In [5]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=32
)

dm.setup()
print(dm)

{Train dataloader: size=24648}
{Validation dataloader: size=2728}
{Test dataloader: size=6849}
{Predict dataloader: None}


## Inter order random walk

In [6]:
import torch
from torch_cluster import random_walk


def uniform_random_walk(edge_index, nodes, batch_size, num_samples,timesteps,length):
    source, target = edge_index[0], edge_index[1]
    
    num_nodes = len(nodes)
    nodes = nodes.repeat(batch_size * num_samples)

    walks, eids = random_walk(row=source,
                              col=target,
                              start=nodes,
                              walk_length=length,
                              return_edge_indices = True,
                              num_nodes=num_nodes)
    
    walks = walks.view(batch_size, num_samples, num_nodes, length+1)
    eids = eids.view(batch_size, num_samples, num_nodes, length)

    walks = walks.unsqueeze(2).repeat(1,1,timesteps,1,1)
    eids = eids.unsqueeze(2).repeat(1,1,timesteps,1,1)
    
    return walks, eids


def uniqueness(walk):
    walk_equal = walk.unsqueeze(-1) == walk.unsqueeze(-2)
    # (1 * walk_equal) -- > bool to int such that can use argmax
    walk_equal = (1 * walk_equal).argmax(dim=-1)
    return walk_equal



## Model

In [7]:
class Embedding(nn.Module):
    def __init__(self, input_size,
                 hidden_size=32,
                 kernel_size=(4,1),
                 stride=(2,1),
                num_samples=5,
                rw_length=5):
        super(Embedding, self).__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.conv = nn.Conv2d(
            in_channels=input_size, 
            out_channels=hidden_size, 
            kernel_size=kernel_size, 
            stride=stride
            )
        self.num_samples = num_samples
        self.rw_length = rw_length

    # def _gather_walk_features(self, features, walks):
    #     """Fully vectorized feature gathering without loops"""
    #     # Create batch indices tensor [batch_size, 1, 1, 1, 1]
    #     batch_indices = torch.arange(walks.shape[0], device=walks.device).view(-1, 1, 1, 1, 1)
    #     print(f"Memory 3: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    #     # Expand batch_indices to match walks shape
    #     batch_indices = batch_indices.expand_as(walks)
    #     print(f"Memory 4: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    #     print('features',features.shape)
    #     print('walks',walks.shape)

    #     features torch.Size([64, 12, 1334, 1])
    #     walks torch.Size([64, 5, 12, 207, 6])
        
    #     # Gather features using advanced indexing
    #     # The result shape will be [batch_size, num_samples, num_nodes, length, feature_dim]
    #     return features[batch_indices, walks]

    def _gather_walk_features(self, features, walks):
        """Process on CPU then transfer back to GPU"""
        # Move tensors to CPU for processing
        cpu_features = features #--> batch_size, timestep, num_simplices, feature
        cpu_walks = walks #--> batch_size, num_random_walk_sample,timestep,num_nodes,random_walk_length

        # cpu_features torch.Size([64, 12, 1334, 1])
        # cpu_walks torch.Size([64, 5, 12, 207, 6])

        features = rearrange(cpu_features, 'b t n f -> b 1 t n 1 f')
        walks = rearrange(walks, 'b s t n l -> b s t n l 1')

        gathered_features = torch.gather(features.expand(-1, walks.shape[1], -1, -1, walks.shape[4], -1),
                                         3,
                                         walks.expand(-1, -1, -1, -1, -1, cpu_features.shape[3]))

        
        # Final shape: [batch_size, num_samples, timestep, num_nodes, length, feature_dim]
        # If you want to remove the timestep dimension:
        # output = rearrange(gathered_features, 'b s t n l f -> b s n l f')
        
        # Transfer back to GPU
        return gathered_features.to(walks.device)

    def forward(self,x,edge_index):
        # x --> [batch_size, time_steps, num_nodes, features]
        batch_size, T, num_nodes, features = x.shape
        walks, eids = uniform_random_walk(
            edge_index=edge_index.to('cuda'), 
            nodes=torch.arange(num_nodes).to('cuda'), 
            batch_size = batch_size,
            num_samples=self.num_samples,
            timesteps = T,
            length=self.rw_length
        ) # -- > [batch_size, num_samples, timesteps, num_nodes, length]
        # torch.Size([32, 1, 12, 207, 4])
        uniqueness_walk = uniqueness(walks)
        walks, uniqueness_walk = walks.flip(-1), uniqueness_walk.flip(-1)
        uniqueness_walk = uniqueness_walk / uniqueness_walk.shape[-1]
        uniqueness_walk = uniqueness_walk * math.pi * 2.0
        uniqueness_walk = torch.cat(
            [
                uniqueness_walk.sin().unsqueeze(-1),
                uniqueness_walk.cos().unsqueeze(-1),
            ],
            dim=-1,
        )

        # Gather features for each node in the walks
        # [batch_size, num_samples, timesteps, num_nodes, length, feature_dim]
        gathered_features = self._gather_walk_features(x, walks)
        # print(x[0,:,walks[0,0,0,-1,0],0])
        # print(x[0,0,walks[0,0,0,-1,0],0])
        # print(gathered_features[0,0,0,walks[0,0,0,-1,0],-1,0])

        # [batch_size, num_samples, timesteps, num_nodes, length, 1, feature_dim]
        # print(f"Memory 2: {torch.cuda.memory_allocated()/1e9:.2f} GB")  
        gathered_features = torch.concat([gathered_features, uniqueness_walk], dim = -1)
        gathered_features = gathered_features.unsqueeze(-2)
        batch_size, num_samples, timesteps, num_nodes, length, _, feature_dim = gathered_features.shape
        # print(f"Memory 3: {torch.cuda.memory_allocated()/1e9:.2f} GB")  
        

        # [batch_size, num_samples, timesteps, num_nodes, length, 1, feature_dim] 
        # -> [batch_size * num_samples * num_nodes * feature_dim, 1, timesteps, length] 
        # print('before',gathered_features[0,0,0,walks[0,0,0,-1,0],-1,0])
        gathered_features = rearrange(gathered_features, 'b s t n l d f -> (b s n f) d t l')
        gathered_features = F.pad(gathered_features,
                          pad=(0, self.kernel_size[1]-self.stride[1],
                               0, self.kernel_size[0]-self.stride[0]),
                          mode='replicate')
        # print(f"Memory 4: {torch.cuda.memory_allocated()/1e9:.2f} GB")  

        x_emb = self.conv(gathered_features)# -> [batch_size * num_samples * num_nodes * feature_dim, D, timesteps_reduce, length_reduce]
        
        x_emb = rearrange(x_emb, '(b s n f) d t l -> b s n f d t l',
                          b=batch_size, s=num_samples, n=num_nodes, f=feature_dim)
        # [batch_size * num_samples * num_nodes * feature_dim, D, timesteps_reduce, length_reduce] 
        # -> [batch_size , num_samples , num_nodes , feature_dim, D, timesteps_reduce, length_reduce]
        # print(f"Memory 5: {torch.cuda.memory_allocated()/1e9:.2f} GB")  
        return x_emb

In [8]:
class ModernTCNBlock(nn.Module):
    def __init__(self, num_nodes, M, D, kernel_size=(4,2), r=1.):
        super(ModernTCNBlock, self).__init__()
        self.dw_conv = nn.Conv2d(
            in_channels=M*D, 
            out_channels=M*D, 
            kernel_size=kernel_size,
            groups=M*D,
            padding='same'
            )  
        self.bn = nn.InstanceNorm2d(M*D)

        self.convffn1 = nn.Sequential(
            nn.Conv2d(
            in_channels=M*D, 
            out_channels=M*D, 
            kernel_size=1,
            groups=M
            ),
            nn.SiLU()
            
        )
        self.convffn2 = nn.Sequential(
            nn.Conv2d(
            in_channels=M*D, 
            out_channels=M*D, 
            kernel_size=1,
            groups=D
            ),
            nn.SiLU()
        )
        # self.pw_con1 = nn.Conv2d(
        #     in_channels=M*D, 
        #     out_channels=M*D, 
        #     kernel_size=1,
        #     groups=M,
        #     # padding='same'
        #     )
        # self.pw_con2 = nn.Conv2d(
        #     in_channels=M*D, 
        #     out_channels=M*D, 
        #     kernel_size=1,
        #     groups=D,
        #     # padding='same'
        #     )

    def forward(self, x_emb):
        # x_emb -> [batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce, length_reduce]
        batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce, length_reduce = x_emb.shape
        
        # First rearrangement
        x = rearrange(x_emb, 'b s n f d t l -> (b s n) (f d) t l')
        
        # Apply convolution
        x = self.dw_conv(x)
        x = self.bn(x)
        
        x = self.convffn1(x)
        
        x = rearrange(x, '(b s n) (f d) t l -> (b s) n f d t l',
                      b=batch_size, s=num_samples, 
                      n=num_nodes, f=feature_dim, d=D)
        
        x = rearrange(x, '(b s) n f d t l -> (b s n) (d f) t l',
                      b=batch_size, s=num_samples)
        
        x = self.convffn2(x)
        
        x = rearrange(x, '(b s n) (d f) t l -> b s d n f t l',
                      b=batch_size, s=num_samples, d=D,
                      n=num_nodes, f=feature_dim)
        
        # [batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce, length_reduce]
        x = rearrange(x, 'b s d n f t l -> b s n f d t l')
        
        # residual connection
        out = x + x_emb
        
        return out

In [9]:
from einops.layers.torch import Rearrange
class ModernTCN(nn.Module):
    def __init__(self, input_size, num_nodes,windows, horizon,rw_sample, rw_length, hidden_size=32, kernel_size=(2,2), stride=(1,1), r=1, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        windows_patches = windows // stride[0]
        rw_length_patches = (rw_length+1) // stride[1]
        
        self.embed_layer = Embedding(input_size, hidden_size,kernel_size = kernel_size,
                                     stride = stride,num_samples=rw_sample,rw_length=rw_length)

        self.backbone = nn.ModuleList([ModernTCNBlock(num_nodes=num_nodes, M=input_size+2,
                                                      D=hidden_size, kernel_size=kernel_size,
                                                      r=r) for _ in range(num_layers)])

        
        # self.head = nn.Linear((input_size+2)*hidden_size*windows_patches*rw_length_patches, horizon)
        # self.head = nn.Sequential(
        #     nn.Linear((input_size+2)*hidden_size*windows_patches*rw_length_patches, hidden_size),
        #     nn.GELU(),
        #     nn.Linear(hidden_size, horizon)
            
        # )

        out_channels = (hidden_size // rw_sample) * rw_sample  # Make divisible

        self.head = nn.Sequential(
            nn.Conv1d(rw_sample*(input_size+2)*hidden_size*windows_patches*rw_length_patches, out_channels,
                      kernel_size=1,
                      groups=rw_sample),
            nn.SiLU(),
            Rearrange('b f n -> b n f'),
            nn.Linear(out_channels, horizon)
            
        )

    def forward(self, x, edge_index):
        # x --> [batch_size, time_steps, num_nodes, features]
        x_emb = self.embed_layer(x, edge_index)

        for i in range(self.num_layers):
            x_emb = self.backbone[i](x_emb)


        # Flatten
        # [batch_size , num_samples , num_nodes , feature_dim, D, timesteps_reduce, length_reduce]
        # x_emb = x_emb.mean(1) # [batch_size , num_nodes , feature_dim, D, timesteps_reduce, length_reduce]
        z = rearrange(x_emb, 'b s n f d t l -> b (s f d t l) n')
        pred = self.head(z)

        # pred = rearrange(pred, 'b n f h -> b h n f')
        pred = rearrange(pred, 'b n h -> b h n')

        # RuntimeError: Predictions and targets are expected to have the same shape, 
        # but got torch.Size([64, 207, 1, 12]) and torch.Size([64, 12, 207, 1]).
        # return pred[:,:,:,0].unsqueeze(-1)
        return pred.unsqueeze(-1)
    

In [10]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
    'mae': torch_metrics.MaskedMAE(),
    'mse': torch_metrics.MaskedMSE(),
    'mae_step_1': torch_metrics.MaskedMAE(at=0),
   'mae_step_2': torch_metrics.MaskedMAE(at=2),
   'mae_step_3': torch_metrics.MaskedMAE(at=5),
   'mae_step_4': torch_metrics.MaskedMAE(at=11)
}


model = ModernTCN(input_size=1,hidden_size = 16,num_nodes=torch_dataset.n_nodes,
                  windows=12, horizon=12, rw_sample=10, rw_length=3,num_layers=5)


def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}_with_graph_with_InstanceNorm_with_fix_timestep"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)


In [11]:
from torch.optim.lr_scheduler import MultiStepLR
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-3
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)
# 'momentum':0.9,
#                  'nesterov':True

In [12]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f'model_checkpoint/{dataset.name}/{model.__class__.__name__}',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
    verbose=True,
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=30,
        mode='min',
        min_delta = 0.03
    )

trainer = Trainer(
        max_epochs=200,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[1],
        gradient_clip_val=5,
       callbacks=[checkpoint_callback, early_stop_callback],
      # default_root_dir="logs",
        # profiler=profiler,
        precision = '16-mixed',
        check_val_every_n_epoch = 5,
        logger=logger  
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [13]:
trainer.fit(predictor, datamodule=dm)

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MetrLA/ModernTCN exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | ModernTCN        | 29.5 K | train
-----------------------------------------------------------
29.5 K    Trainable params
0         Non-trainable params
29.5 K    Total params
0.118     Total estimated model params size (MB)
76        Modules in train mode
0         Modules in eval mode
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/py

Training: |                                                                                                   …

Arguments ['edge_weight', 'u'] are filtered out. Only args ['edge_index', 'x'] are forwarded to the model (ModernTCN).
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/torch/nn/modules/conv.py:454: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ../aten/src/ATen/native/Convolution.cpp:1031.)
  return F.conv2d(input, weight, bias, self.stride,
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Validation: |                                                                                                 …

Epoch 4, global step 750: 'val_mae' reached 3.23645 (best 3.23645), saving model to '/netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MetrLA/ModernTCN/epoch=4-step=750-v3.ckpt' as top 1


...

Epoch 189, global step 28500: 'val_mae' was not in top 1


Validation: |                                                                                                 …

Epoch 194, global step 29250: 'val_mae' was not in top 1


Validation: |                                                                                                 …

Epoch 199, global step 30000: 'val_mae' reached 3.06414 (best 3.06414), saving model to '/netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MetrLA/ModernTCN/epoch=199-step=30000.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=200` reached.


In [14]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MetrLA/ModernTCN/epoch=199-step=30000.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MetrLA/ModernTCN/epoch=199-step=30000.ckpt
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Testing: |                                                                                                    …

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     3.530792236328125     │
│         test_mae          │     3.727494716644287     │
│      test_mae_step_1      │    2.4141275882720947     │
│      test_mae_step_2      │    3.0456647872924805     │
│      test_mae_step_3      │    3.7043583393096924     │
│      test_mae_step_4      │     4.75387716293335      │
│         test_mse          │    56.132503509521484     │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 3.727494716644287,
  'test_mae_step_1': 2.4141275882720947,
  'test_mae_step_2': 3.0456647872924805,
  'test_mae_step_3': 3.7043583393096924,
  'test_mae_step_4': 4.75387716293335,
  'test_mse': 56.132503509521484,
  'test_loss': 3.530792236328125}]

In [ ]:
print(profiler.summary())

In [ ]:
cpu_features torch.Size([64, 12, 1334, 1]) --> batch_size, timestep, num_simplices, feature
cpu_walks torch.Size([64, 5, 12, 207, 6])  --> batch_size, num_random_walk_sample,timestep,num_nodes,random_walk_length

In [ ]:
torch.rand([64, 12, 1334, 3])[:,:,:,0].unsqueeze(-1).shape